# Phase P4: Reversible Variant Pilot & Selection
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/thinkTinkerGreen/train_with_reversibility/blob/main/notebooks/03_reversible_pilot.ipynb)


In [ ]:
import os, sys, json, subprocess
SMOKE = os.environ.get("REVLLM_SMOKE", "0") == "1"
IN_COLAB = "google.colab" in sys.modules

REPO_URL = "https://github.com/thinkTinkerGreen/train_with_reversibility.git"
REPO_REF = "main"

if IN_COLAB and not SMOKE:
    from google.colab import drive
    drive.mount('/content/drive')
    os.makedirs('/content/drive/MyDrive/revllm/data', exist_ok=True)
    os.makedirs('/content/drive/MyDrive/revllm/runs', exist_ok=True)
    os.makedirs('/content/drive/MyDrive/revllm/probes', exist_ok=True)

    repo_dir = "/content/train_with_reversibility"
    if not os.path.exists(repo_dir):
        print(f"Cloning {REPO_URL} into {repo_dir}...")
        subprocess.run(["git", "clone", REPO_URL, repo_dir], check=True)
    else:
        print(f"Repository already present at {repo_dir}. Fetching updates...")
        subprocess.run(["git", "-C", repo_dir, "fetch", "--all"], check=False)
        subprocess.run(["git", "-C", repo_dir, "reset", "--hard", f"origin/{REPO_REF}"], check=False)

    os.chdir(repo_dir)

    if repo_dir not in sys.path:
        sys.path.insert(0, repo_dir)
    src_dir = os.path.join(repo_dir, "src")
    if src_dir not in sys.path:
        sys.path.insert(0, src_dir)

    # Ensure dependencies and package are installed in fresh Colab runtimes
    deps = ["hatchling", "pyyaml", "tokenizers", "huggingface-hub", "datasets", "tqdm"]
    subprocess.run([sys.executable, "-m", "pip", "install", "-q"] + deps, check=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--no-deps", "-e", "."], check=True)
print(f"Current working directory: {os.getcwd()}")
print(f"Execution mode: {'Smoke (CPU/CI)' if SMOKE else 'Colab GPU'}")


In [ ]:
from revllm.config import ModelConfig, RunConfig, TrainConfig
from revllm.train import train_run

# Restore dataset from Drive if needed
if IN_COLAB and not SMOKE:
    os.makedirs("data", exist_ok=True)
    for fn in ["train.bin", "val.bin", "tokenizer.json"]:
        drive_fp = f"/content/drive/MyDrive/revllm/data/{fn}"
        if os.path.exists(drive_fp) and not os.path.exists(f"data/{fn}"):
            import shutil
            shutil.copy2(drive_fp, f"data/{fn}")

model_cfg = ModelConfig.from_yaml("configs/model.yaml")
train_cfg = TrainConfig.from_yaml("configs/train.yaml")
# Pilot trains 10M tokens (or 5k tokens in smoke)
train_cfg.training.total_tokens = 5000 if SMOKE else 10_000_000

candidates = ["midpoint", "midpoint_theta", "leapfrog"]
pilot_results = []

for cand in candidates:
    print(f"\n=== Running Pilot for Variant: {cand} ===")
    run_cfg = RunConfig(
        run_id=f"pilot_{cand}",
        arch=cand,
        h=0.5 if cand == "midpoint" else 1.0,
        batch_size=16,
        seed=1337,
        compile=False,
    )
    res = train_run(
        model_cfg=model_cfg,
        train_cfg=train_cfg,
        run_cfg=run_cfg,
        train_data_path="data/train.bin",
        val_data_path="data/val.bin",
        local_output_dir=f"results/runs/pilot_{cand}",
        smoke_mode=SMOKE,
    )
    pilot_results.append({
        "arch": cand,
        "val_loss": res.get("final_val_loss"),
        "stability": "PASS" if not res.get("final_val_loss") != res.get("final_val_loss") else "FAIL",
        "recon_max_abs": res.get("recon_err_max_abs"),
        "tps": res.get("tokens_per_s_median"),
    })

# Mechanical winner selection rule
survivors = [c for c in pilot_results if c["stability"] == "PASS" and (c["recon_max_abs"] is None or c["recon_max_abs"] <= 1e-3)]
survivors.sort(key=lambda x: (x["val_loss"] if x["val_loss"] is not None else 999.0))
winner = survivors[0]["arch"] if survivors else "midpoint"

selection_data = {
    "winner": winner,
    "rationale": f"Surviving candidate with lowest validation loss ({survivors[0]['val_loss']:.4f})" if survivors else "Default fallback",
    "candidates": pilot_results,
}

os.makedirs("results", exist_ok=True)
with open("results/variant_selection.json", "w") as f:
    json.dump(selection_data, f, indent=2)

if IN_COLAB and not SMOKE:
    import shutil
    shutil.copy2("results/variant_selection.json", "/content/drive/MyDrive/revllm/runs/variant_selection.json")

print(f"Pilot Winner Selected: {winner}")
